# ML Practical 6 – Tic-Tac-Toe using Q-Learning

**Aim:** Implement a Q-learning agent, train it against a random opponent, test its win rate, and optionally play against it.

In [ ]:
import numpy as np
import random
from collections import defaultdict

class TicTacToe:
    def __init__(self):
        self.reset()

    def reset(self):
        self.board = [0] * 9
        return self.get_state()

    def get_state(self):
        return tuple(self.board)

    def available_actions(self):
        return [i for i in range(9) if self.board[i] == 0]

    def make_move(self, action, player):
        if action not in self.available_actions():
            return False
        self.board[action] = player
        return True

    def check_winner(self):
        winning_positions = [
            (0, 1, 2), (3, 4, 5), (6, 7, 8),
            (0, 3, 6), (1, 4, 7), (2, 5, 8),
            (0, 4, 8), (2, 4, 6)
        ]
        for a, b, c in winning_positions:
            if self.board[a] != 0 and self.board[a] == self.board[b] == self.board[c]:
                return self.board[a]
        if 0 not in self.board:
            return 0
        return None

    def display(self):
        symbols = {1: "X", -1: "O", 0: " "}
        for i in range(0, 9, 3):
            print(" | ".join(symbols[self.board[j]] for j in range(i, i + 3)))
            if i < 6:
                print("---------")

In [ ]:
class QLearningAgent:
    def __init__(self, learning_rate=0.1, discount_factor=0.9,
                 epsilon=1.0, epsilon_decay=0.9995, epsilon_min=0.01):
        self.q_table = defaultdict(lambda: np.zeros(9))
        self.alpha = learning_rate
        self.gamma = discount_factor
        self.epsilon = epsilon
        self.epsilon_decay = epsilon_decay
        self.epsilon_min = epsilon_min

    def choose_action(self, state, actions, training=True):
        if training and random.random() < self.epsilon:
            return random.choice(actions)
        values = self.q_table[state]
        best_value = max(values[a] for a in actions)
        best_actions = [a for a in actions if values[a] == best_value]
        return random.choice(best_actions)

    def update(self, state, action, reward, next_state, next_actions, done):
        current_q = self.q_table[state][action]
        future_q = max((self.q_table[next_state][a] for a in next_actions), default=0)
        target = reward if done else reward + self.gamma * future_q
        self.q_table[state][action] += self.alpha * (target - current_q)

    def decay_epsilon(self):
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

def random_opponent(game):
    actions = game.available_actions()
    return random.choice(actions) if actions else None

In [ ]:
def train_agent(agent, episodes=50000):
    print("Training the Tic-Tac-Toe agent...")
    for episode in range(episodes):
        game = TicTacToe()
        state = game.reset()

        while True:
            actions = game.available_actions()
            if not actions:
                break

            action = agent.choose_action(state, actions, training=True)
            game.make_move(action, 1)
            result = game.check_winner()
            next_state = game.get_state()

            if result == 1:
                agent.update(state, action, 1, next_state, [], True)
                break
            if result == 0:
                agent.update(state, action, 0, next_state, [], True)
                break

            opponent_action = random_opponent(game)
            game.make_move(opponent_action, -1)
            result = game.check_winner()
            next_state = game.get_state()

            if result == -1:
                agent.update(state, action, -1, next_state, [], True)
                break
            if result == 0:
                agent.update(state, action, 0, next_state, [], True)
                break

            next_actions = game.available_actions()
            agent.update(state, action, 0, next_state, next_actions, False)
            state = next_state

        agent.decay_epsilon()
        if (episode + 1) % 5000 == 0:
            print(f"Episode {episode + 1}: Epsilon = {agent.epsilon:.4f}")

In [ ]:
def test_agent(agent, games=1000):
    wins = losses = draws = 0
    for _ in range(games):
        game = TicTacToe()
        while True:
            state = game.get_state()
            actions = game.available_actions()
            if not actions:
                draws += 1
                break

            action = agent.choose_action(state, actions, training=False)
            game.make_move(action, 1)
            result = game.check_winner()
            if result == 1:
                wins += 1
                break
            if result == 0:
                draws += 1
                break

            opponent_action = random_opponent(game)
            game.make_move(opponent_action, -1)
            result = game.check_winner()
            if result == -1:
                losses += 1
                break
            if result == 0:
                draws += 1
                break

    print("\nTesting Results")
    print("Total Games:", games)
    print("Wins:", wins)
    print("Losses:", losses)
    print("Draws:", draws)
    print("Win Rate:", round(wins / games * 100, 2), "%")

In [ ]:
def display_numbered_board():
    print(" 1 | 2 | 3")
    print("-----------")
    print(" 4 | 5 | 6")
    print("-----------")
    print(" 7 | 8 | 9")

def play_against_ai(agent):
    game = TicTacToe()
    print("You are O; AI is X.")
    display_numbered_board()

    while True:
        game.display()
        result = game.check_winner()
        if result == 1:
            print("AI wins!")
            break
        if result == -1:
            print("You win!")
            break
        if result == 0:
            print("Game draw!")
            break

        while True:
            try:
                action = int(input("Enter your position (1-9): ")) - 1
                if action in game.available_actions():
                    break
                print("Choose an empty position.")
            except ValueError:
                print("Enter a number between 1 and 9.")

        game.make_move(action, -1)
        result = game.check_winner()
        if result is not None:
            continue

        state = game.get_state()
        ai_action = agent.choose_action(state, game.available_actions(), training=False)
        game.make_move(ai_action, 1)
        print("AI chose position:", ai_action + 1)

In [ ]:
agent = QLearningAgent()
train_agent(agent, episodes=50000)
test_agent(agent, games=1000)

choice = input("Do you want to play against the AI? (yes/no): ").lower()
if choice == "yes":
    play_against_ai(agent)
else:
    print("Program completed successfully.")

In [ ]:
# ========== ML PRACTICAL 6: TIC-TAC-TOE USING Q-LEARNING ==========

# AIM: Train an AI to play Tic-Tac-Toe using Q-Learning.
# FLOW: Create Game -> Choose Action -> Reward -> Update Q -> Train -> Test

# THEORY:
# Reinforcement Learning (RL) = learning through rewards and penalties.
# Agent = AI player (X); Environment = Tic-Tac-Toe game.
# State = current board; Action = placing X in an empty cell.
# Reward = feedback (+1 win, -1 loss, 0 draw/ongoing).
# Q-Learning = learns the best actions using Q-values.
# Q-value = expected future reward for an action in a state.
# Q-Table = stores Q-values for different state-action pairs.
# Exploration = trying random moves; Exploitation = using best moves.
# Epsilon = probability of exploration.
# Alpha = learning rate; Gamma = discount factor.

# Q-LEARNING FORMULA:
# Q(s,a) = Q(s,a) + alpha * [r + gamma*max(Q(s',a')) - Q(s,a)]
# s = current state; a = action; r = reward.
# s' = next state; alpha = learning rate; gamma = discount factor.
# For terminal states, future Q-value = 0.

# LIBRARIES:
# numpy(np) = numerical arrays and calculations.
# random = selects random actions.
# defaultdict = dictionary with automatic default values.

# CLASS TicTacToe:
# __init__() = constructor; initializes game using reset().
# reset() = creates empty board [0]*9 and returns state.
# 0 = empty; 1 = AI (X); -1 = opponent (O).
# get_state() = returns board as tuple (immutable).
# available_actions() = list of empty cell indices (0-8).
# make_move(action,player) = places mark if move is valid.
# Returns True for valid move, False for invalid move.
# check_winner() = checks 8 possible winning combinations.
# Returns 1 = X wins, -1 = O wins, 0 = draw, None = ongoing.
# display() = prints board using X, O and blank spaces.
# join() = joins symbols with separators.
# range(0,9,3) = rows starting at 0, 3, 6.

# CLASS QLearningAgent:
# __init__() = initializes learning parameters and Q-table.
# defaultdict(lambda:np.zeros(9)) = 9 zero Q-values per new state.
# learning_rate=0.1 = speed of learning (alpha).
# discount_factor=0.9 = importance of future rewards (gamma).
# epsilon=1.0 = starts with 100% random exploration.
# epsilon_decay=0.9995 = reduces exploration each episode.
# epsilon_min=0.01 = minimum 1% exploration during training.

# choose_action(state,actions,training=True):
# state = current board; actions = valid moves.
# training=True = exploration allowed; False = best known moves.
# random.random() = random value between 0 and 1.
# random.choice(actions) = randomly selects a valid move.
# max() = finds highest Q-value among valid actions.
# best_actions = moves sharing highest Q-value.
# Returns one randomly selected best action.

# update(state,action,reward,next_state,next_actions,done):
# current_q = stored Q-value for selected action.
# future_q = highest Q-value of valid next actions.
# default=0 = use zero if no next actions exist.
# done=True = game finished; no future reward.
# target = reward + gamma*future_q (if not finished).
# Q-value updated using alpha*(target-current_q).

# decay_epsilon():
# epsilon *= epsilon_decay = gradually reduces random moves.
# max(epsilon_min,...) = prevents epsilon falling below 0.01.

# RANDOM OPPONENT:
# random_opponent(game) = selects a random empty position.
# Returns None if no empty positions exist.

# TRAINING:
# train_agent(agent,episodes=50000) = trains for 50,000 games.
# for episode in range(episodes) = repeats training games.
# AI chooses X move, then random opponent chooses O move.
# check_winner() checks result after each move.
# AI win = +1; AI loss = -1; draw = 0.
# agent.update() = improves Q-values from experience.
# state=next_state = moves to next decision state.
# break = ends loop; agent.decay_epsilon() = reduces exploration.
# (episode+1)%5000==0 = prints progress every 5000 games.
# :.4f = displays epsilon to 4 decimal places.

# TESTING:
# test_agent(agent,games=1000) = tests AI in 1000 games.
# training=False = selects best known moves (no exploration).
# wins/losses/draws = count game outcomes.
# Win Rate = (wins / total games)*100.
# round(value,2) = rounds result to 2 decimal places.
# Testing does not update Q-values.

# HUMAN VS AI:
# display_numbered_board() = shows positions 1 to 9.
# play_against_ai(agent) = allows human to play against AI.
# game.display() = shows current board.
# input() = takes user move; int() = converts text to integer.
# -1 = converts positions 1-9 to indices 0-8.
# try/except ValueError = handles invalid numerical input.
# continue = starts next loop iteration.
# AI uses choose_action(...,training=False) to select move.

# MAIN PROGRAM:
# QLearningAgent() = creates AI agent.
# train_agent(agent,episodes=50000) = trains AI.
# test_agent(agent,games=1000) = evaluates AI.
# input(...).lower() = reads choice in lowercase.
# choice=="yes" = starts human vs AI game.

# ========== 4 IMPORTANT VIVA QUESTIONS ==========

# Q1. What is Reinforcement Learning? How is it different?
# Ans: RL learns using rewards and penalties.
# Supervised = labeled data; Unsupervised = finds hidden patterns.
# RL = learns through trial, error and feedback.

# Q2. Real-world applications of Reinforcement Learning?
# Ans: Robotics, self-driving research, game AI,
#      resource management and recommendation systems.

# Q3. How does Q-Learning work? What is Q-function?
# Ans: Q-Learning updates Q-values using rewards and
# future rewards to learn the best action.
# Q-function estimates expected future reward of an action.

# Q4. What is the role of reward in RL?
# Ans: Reward tells the agent whether an action was good or bad,
# guiding it toward actions with higher future rewards.

# FINAL VIVA EXPLANATION:
# "We create a Tic-Tac-Toe environment and a Q-Learning agent.
# The agent learns by playing 50,000 games against a random
# opponent. It receives +1 for winning, -1 for losing and
# 0 for drawing. Q-values improve through repeated games.
# Finally, we test the agent over 1,000 games, calculate
# its win rate and optionally allow a human to play."